# Technical Note P8: SciPy Sparse Matrices and Sparse Solvers

**Wook Kang · Revised English edition · Version 1.0.0 · 3 October 2026**

This note connects a mathematical task to a numerical object, an implementation, and evidence about the result. It is an instructional computational document. Demonstration parameters are illustrative unless explicitly stated otherwise.

Run from a fresh kernel in cell order. See [the series README](../README.md) for setup and [editorial and verification notes](../docs/EDITORIAL_NOTES.md) for scope and limitations. The English prose has been reorganized and expanded from the supplied Korean notebook; this is an adapted edition rather than a line-by-line translation.

**Reading question:** What is given, what is unknown, what operation relates them, and what independent check can assess the output?

## Core mathematical structure

$$
\boxed{
\text{local interaction}
\rightarrow
\text{sparse matrix}
}
$$

$$
\boxed{
\text{dense}
\rightarrow
\text{sparse}
\rightarrow
\text{iterative / matrix-free}
}
$$

In [1]:
# Reproducible display and scale-aware comparison.
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display
plt.rcParams.update({"figure.dpi": 110, "axes.grid": False})
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
def scaled_allclose(a, b, rtol=1e-12, atol=None, **kwargs):
    """Compare numerical examples using an absolute tolerance scaled to b."""
    a, b = np.asarray(a), np.asarray(b)
    scale = float(np.max(np.abs(b))) if b.size else 0.0
    if atol is None:
        atol = 1e-14 * max(scale, np.finfo(float).tiny)
    return np.allclose(a, b, rtol=rtol, atol=atol, **kwargs)


## 1. Why PDE matrices are sparse

Local stencils or finite elements couple each unknown to relatively few neighbours. Boundary constraints and nonlocal models can change this pattern.

$$
-\frac{u_{i-1}-2u_i+u_{i+1}}{\Delta x^2}=f_i
$$

$$
u_{i-1},\quad u_i,\quad u_{i+1}
$$

In [2]:
import numpy as np
from scipy import sparse
from scipy.sparse import linalg as spla

N = 8
main = 2*np.ones(N)
off = -1*np.ones(N-1)

A = sparse.diags([off, main, off], offsets=[-1, 0, 1], format="csr")

print(A.toarray())
print("shape =", A.shape)
print("nnz =", A.nnz)

[[ 2. -1.  0.  0.  0.  0.  0.  0.]
 [-1.  2. -1.  0.  0.  0.  0.  0.]
 [ 0. -1.  2. -1.  0.  0.  0.  0.]
 [ 0.  0. -1.  2. -1.  0.  0.  0.]
 [ 0.  0.  0. -1.  2. -1.  0.  0.]
 [ 0.  0.  0.  0. -1.  2. -1.  0.]
 [ 0.  0.  0.  0.  0. -1.  2. -1.]
 [ 0.  0.  0.  0.  0.  0. -1.  2.]]
shape = (8, 8)
nnz = 22


## 2. Nonzero storage

Nnz counts stored entries, which can include explicit zeros and duplicates before consolidation. Sparse storage is useful when the stored pattern is sufficiently small.

```python
A.nnz
```

$$
3N
$$

In [3]:
for N in [10, 100, 1000, 10000]:
    nnz = 3*N - 2
    dense_entries = N*N
    print(f"N={N:5d}  dense={dense_entries:12d}  sparse nnz≈{nnz:8d}")

N=   10  dense=         100  sparse nnz≈      28
N=  100  dense=       10000  sparse nnz≈     298
N= 1000  dense=     1000000  sparse nnz≈    2998
N=10000  dense=   100000000  sparse nnz≈   29998


## 3. Sparse formats

COO supports construction, CSR supports row operations and products, and CSC supports column-oriented factorization interfaces. Convert at deliberate boundaries.

| Format | Strength | Typical use |  
| --- | --- | --- |  
| COO | Simple assembly | triplet assembly |  
| CSR | Efficient row operations and matvec | solver, PDE operator |  
| CSC | column operation / factorization | direct solver |  
| LIL | incremental row-wise editing | matrix construction |  
| DOK | dictionary-style insertion | irregular assembly |  
| DIA | diagonal structure | banded operator |  
| BSR | block structure | vector PDE / FEM blocks |

$$
\boxed{
\text{assemble in COO/LIL}
\rightarrow
\text{convert to CSR/CSC}
\rightarrow
\text{solve}
}
$$

## 4. COO assembly

Store row indices, column indices, and values. Duplicate coordinates are combined when converting to a canonical compressed format, which suits assembly contributions.

$$
(row,\ column,\ value)
$$

In [4]:
rows = np.array([0, 0, 1, 1, 2])
cols = np.array([0, 1, 0, 1, 2])
vals = np.array([2.0, -1.0, -1.0, 2.0, 3.0])

A_coo = sparse.coo_matrix((vals, (rows, cols)), shape=(3, 3))

print(A_coo)
print(A_coo.toarray())

<COOrdinate sparse matrix of dtype 'float64'
	with 5 stored elements and shape (3, 3)>
  Coords	Values
  (0, 0)	2.0
  (0, 1)	-1.0
  (1, 0)	-1.0
  (1, 1)	2.0
  (2, 2)	3.0
[[ 2. -1.  0.]
 [-1.  2.  0.]
 [ 0.  0.  3.]]


In [5]:
rows = np.array([0, 0, 0])
cols = np.array([0, 0, 1])
vals = np.array([1.0, 2.0, 5.0])

A_dup = sparse.coo_matrix((vals, (rows, cols)), shape=(2, 2))
A_dup.sum_duplicates()

print(A_dup.toarray())

[[3. 5.]
 [0. 0.]]


## 5. CSR

Compressed sparse row storage is well suited to matrix-vector products. Repeated structural changes are comparatively expensive.

In [6]:
A_csr = A_coo.tocsr()

print(A_csr)
print("data    =", A_csr.data)
print("indices =", A_csr.indices)
print("indptr  =", A_csr.indptr)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 5 stored elements and shape (3, 3)>
  Coords	Values
  (0, 0)	2.0
  (0, 1)	-1.0
  (1, 0)	-1.0
  (1, 1)	2.0
  (2, 2)	3.0
data    = [ 2. -1. -1.  2.  3.]
indices = [0 1 0 1 2]
indptr  = [0 2 4 5]


$$
\boxed{
\text{nonzero value}
+
\text{where it lives}
}
$$

## 6. CSC

Compressed sparse column storage is commonly used for sparse LU and ILU. Avoid repeatedly converting inside an iteration.

In [7]:
A_csc = A_csr.tocsc()

print("CSR format:", A_csr.format)
print("CSC format:", A_csc.format)

CSR format: csr
CSC format: csc


## 7. Diagonal stencils

Diags builds operators from their diagonal bands. Decide operator sign, unknown count, and boundary treatment before supplying coefficients.

```python
sparse.diags
```

In [8]:
N = 6

A = sparse.diags(
    diagonals=[
        -np.ones(N-1),
         2*np.ones(N),
        -np.ones(N-1),
    ],
    offsets=[-1, 0, 1],
    format="csr",
)

print(A.toarray())

[[ 2. -1.  0.  0.  0.  0.]
 [-1.  2. -1.  0.  0.  0.]
 [ 0. -1.  2. -1.  0.  0.]
 [ 0.  0. -1.  2. -1.  0.]
 [ 0.  0.  0. -1.  2. -1.]
 [ 0.  0.  0.  0. -1.  2.]]


## 8. Dense and sparse memory

Dense storage grows with the square of unknown count. Sparse memory also includes index arrays, and factorization fill-in can exceed the original operator's storage.

$$
8N^2\ \text{bytes}
$$

$$
\boxed{
O(N^2)
\quad\text{vs}\quad
O(\mathrm{nnz})
}
$$

In [9]:
N = 2000

A_sparse = sparse.diags(
    [-np.ones(N-1), 2*np.ones(N), -np.ones(N-1)],
    [-1, 0, 1],
    format="csr",
)

dense_bytes = N*N*8
sparse_bytes = A_sparse.data.nbytes + A_sparse.indices.nbytes + A_sparse.indptr.nbytes

print("dense MB  ≈", dense_bytes/1024**2)
print("sparse MB =", sparse_bytes/1024**2)
print("nnz =", A_sparse.nnz)

dense MB  ≈ 30.517578125
sparse MB = 0.07627487182617188
nnz = 5998


## 9. Sparse matrix-vector products

The at operator evaluates a sparse linear map without making a dense array. Inspect both the mathematical operation and the output shape.

$$
y=Ax
$$

```python
A @ x
```

In [10]:
x = np.arange(A_sparse.shape[0], dtype=float)
y = A_sparse @ x

print("y shape =", y.shape)
print("first values =", y[:5])

y shape = (2000,)
first values = [-1.  0.  0.  0.  0.]


## 10. Sparse direct solves

Spsolve factors and solves a sparse system. A successful return should still be followed by a residual and finite-value check.

$$
Ax=b
$$

```python
spsolve(A,b)
```

In [11]:
N = 20
dx = 1/(N+1)

A = sparse.diags(
    [-np.ones(N-1), 2*np.ones(N), -np.ones(N-1)],
    [-1, 0, 1],
    format="csr",
) / dx**2

b = np.ones(N)

u = spla.spsolve(A, b)

print("solution shape =", u.shape)
print("residual norm =", np.linalg.norm(A @ u - b))

solution shape = (20,)
residual norm = 3.539365928370457e-14


## 11. Reuse a factorization

Reuse a factored operator for several right-hand sides only while its coefficients and constraints remain unchanged.

```python
factorized(A)
```

In [12]:
solve_A = spla.factorized(A.tocsc())

b1 = np.ones(N)
b2 = np.linspace(0, 1, N)

u1 = solve_A(b1)
u2 = solve_A(b2)

print("residual 1 =", np.linalg.norm(A @ u1 - b1))
print("residual 2 =", np.linalg.norm(A @ u2 - b2))

residual 1 = 2.0485946440854968e-14
residual 2 = 9.179503631832733e-15


## 12. Sparse LU

Splu provides a reusable factorization object. The factors can contain substantially more nonzeros than the assembled matrix.

In [13]:
lu = spla.splu(A.tocsc())
u_lu = lu.solve(b)

print("residual =", np.linalg.norm(A @ u_lu - b))

residual = 2.0485946440854968e-14


## 13. Direct and iterative strategies

Direct methods offer predictable solves but can incur fill-in; iterative methods need convergence monitoring and often preconditioning. Compare total cost at the required accuracy.

## 14. Conjugate gradients

CG requires a Hermitian positive-definite operator and a compatible positive-definite preconditioner. A discretized Laplacian may have the wrong sign or a nullspace.

$$
\boxed{
\text{symmetric positive definite}
}
$$

In [14]:
N = 200
dx = 1/(N+1)

A = sparse.diags(
    [-np.ones(N-1), 2*np.ones(N), -np.ones(N-1)],
    [-1, 0, 1],
    format="csr",
) / dx**2

b = np.ones(N)

u_cg, info = spla.cg(A, b, rtol=1e-10, atol=0.0, maxiter=1000)

print("info =", info)
print("residual norm =", np.linalg.norm(A @ u_cg - b))

info = 0
residual norm = 5.352646283143688e-12


## 15. GMRES

GMRES handles general nonsymmetric systems. Restart reduces memory but can slow convergence; inspect the returned info and true residual.

$$
Ax=b
$$

In [15]:
A_ns = A + sparse.diags([0.05*np.ones(N-1)], [1], shape=A.shape, format="csr")

u_gmres, info = spla.gmres(
    A_ns,
    b,
    rtol=1e-10,
    atol=0.0,
    restart=50,
    maxiter=1000,
)

print("info =", info)
print("residual norm =", np.linalg.norm(A_ns @ u_gmres - b))

info = 0
residual norm = 1.4126200328793724e-09


## 16. BiCGSTAB

BiCGSTAB is another nonsymmetric option with different convergence behaviour. Breakdown or stagnation must be treated as failure, not hidden by a finite output vector.

In [16]:
u_bicg, info = spla.bicgstab(
    A_ns,
    b,
    rtol=1e-10,
    atol=0.0,
    maxiter=1000,
)

print("info =", info)
print("residual norm =", np.linalg.norm(A_ns @ u_bicg - b))

info = 0
residual norm = 1.157914814201408e-09


## 17. Select an iterative method

Use matrix structure, spectrum, conditioning, and preconditioner properties to guide the choice. Solver availability does not establish compatibility.

| Matrix structure | Representative solver |  
| --- | --- |  
| SPD | CG |  
| symmetric indefinite | MINRES |  
| general nonsymmetric | GMRES |  
| general nonsymmetric | BiCGSTAB |

## 18. Preconditioning

A preconditioner supplies an approximate inverse action for the solver's interface. It changes convergence behaviour without changing the desired exact solution.

$$
M^{-1}Ax=M^{-1}b
$$

## 19. Jacobi preconditioning

Divide by the diagonal when entries are appropriate and nonzero. For a constant-diagonal operator this is only a scalar rescaling and need not improve conditioning.

$$
M=\operatorname{diag}(A)
$$

$$
M^{-1}v
$$

In [17]:
diagA = A.diagonal()

def jacobi_apply(v):
    return v / diagA

M_jacobi = spla.LinearOperator(A.shape, matvec=jacobi_apply)

u_pc, info = spla.cg(
    A,
    b,
    M=M_jacobi,
    rtol=1e-10,
    atol=0.0,
    maxiter=1000,
)

print("info =", info)
print("residual norm =", np.linalg.norm(A @ u_pc - b))

info = 0
residual norm = 1.1893356284682742e-11


## 20. ILU preconditioning

ILU approximates a factorization and often helps nonsymmetric solvers. A generic ILU need not preserve the symmetry and definiteness required by CG.

$$
A\approx LU
$$

In [18]:
ilu = spla.spilu(A_ns.tocsc())

M_ilu = spla.LinearOperator(
    A_ns.shape,
    matvec=ilu.solve
)

u_ilu, info = spla.gmres(
    A_ns,
    b,
    M=M_ilu,
    rtol=1e-10,
    atol=0.0,
    restart=50,
    maxiter=1000,
)

print("info =", info)
print("residual norm =", np.linalg.norm(A_ns @ u_ilu - b))

info = 0
residual norm = 1.2781966683401787e-11


## 21. LinearOperator

Supply the action Av with an explicit shape and dtype. Matrix-free storage does not remove the need for correct boundaries or a useful preconditioner.

$$
y=Ax
$$

$$
\boxed{
\text{operator}
\neq
\text{explicit matrix}
}
$$

In [19]:
N = 100

def matvec_1d_poisson(x):
    y = np.zeros_like(x)
    y[:] = 2*x
    y[:-1] -= x[1:]
    y[1:]  -= x[:-1]
    return y

Aop = spla.LinearOperator((N, N), matvec=matvec_1d_poisson, dtype=float)

x_test = np.linspace(0, 1, N)
y_test = Aop @ x_test

print("operator output shape =", y_test.shape)

operator output shape = (100,)


## 22. Matrix-free CG

The example applies the same positive one-dimensional stencil without storing its matrix. Check equivalence with an assembled operator on an arbitrary vector.

In [20]:
b = np.ones(N)

u_mf, info = spla.cg(
    Aop,
    b,
    rtol=1e-10,
    atol=0.0,
    maxiter=1000,
)

res = Aop @ u_mf - b

print("info =", info)
print("residual norm =", np.linalg.norm(res))

info = 0
residual norm = 0.0


## 23. COO contributions

Assembly collects local contributions into global indices. Repeated indices must accumulate rather than overwrite.

$$
(row_I,\ col_J,\ value)
$$

In [21]:
rows = []
cols = []
vals = []

N = 5

for e in range(N-1):
    local = np.array([
        [1.0, -1.0],
        [-1.0, 1.0],
    ])
    nodes = [e, e+1]

    for a in range(2):
        for b_ in range(2):
            rows.append(nodes[a])
            cols.append(nodes[b_])
            vals.append(local[a, b_])

K_coo = sparse.coo_matrix((vals, (rows, cols)), shape=(N, N))
K = K_coo.tocsr()

print(K.toarray())

[[ 1. -1.  0.  0.  0.]
 [-1.  2. -1.  0.  0.]
 [ 0. -1.  2. -1.  0.]
 [ 0.  0. -1.  2. -1.]
 [ 0.  0.  0. -1.  1.]]


$$
\boxed{
\text{local matrix}
\rightarrow
\text{global sparse assembly}
}
$$

## 24. Kronecker products in two dimensions

Tie the Kronecker expression to flattening order and coordinate axes. Different valid orderings lead to different-looking but related matrices.

$$
L_{2D}=
I_y\otimes L_x
+
L_y\otimes I_x
$$

In [22]:
Nx, Ny = 5, 4

Lx = sparse.diags(
    [-np.ones(Nx-1), 2*np.ones(Nx), -np.ones(Nx-1)],
    [-1, 0, 1],
    format="csr",
)

Ly = sparse.diags(
    [-np.ones(Ny-1), 2*np.ones(Ny), -np.ones(Ny-1)],
    [-1, 0, 1],
    format="csr",
)

Ix = sparse.eye(Nx, format="csr")
Iy = sparse.eye(Ny, format="csr")

L2 = sparse.kron(Iy, Lx, format="csr") + sparse.kron(Ly, Ix, format="csr")

print("shape =", L2.shape)
print("nnz =", L2.nnz)

shape = (20, 20)
nnz = 82


## 25. Sparse eigenproblems

Eigsh is for Hermitian problems and eigs is for general problems. Requesting a few modes differs from computing an entire dense spectrum.

```python
eigsh
eigs
```

In [23]:
vals, vecs = spla.eigsh(L2, k=3, which="SM")

print("smallest eigenvalues =", vals)

smallest eigenvalues = [0.6499152  1.38196601 1.6499152 ]


## 26. Residual norms

Record absolute and scaled residuals. Handle a zero right-hand-side norm explicitly rather than dividing by zero.

$$
r=Ax-b
$$

$$
\frac{\|r\|}{\|b\|}
$$

In [24]:
b = np.ones(L2.shape[0])

u, info = spla.cg(L2, b, rtol=1e-10, atol=0.0, maxiter=2000)

r = L2 @ u - b

print("info =", info)
print("absolute residual =", np.linalg.norm(r))
print("relative residual =", np.linalg.norm(r)/np.linalg.norm(b))

info = 0
absolute residual = 2.1152589550219202e-15
relative residual = 4.729862813444184e-16


## 27. Tolerances

For these Krylov interfaces, convergence is judged against a combination of relative and absolute residual thresholds. This is an algebraic stopping criterion, not a physical prediction interval.

$$
Ax\approx b
$$

$$
\boxed{
\text{solver tolerance}
\neq
\text{model accuracy}
}
$$

## 28. Format-conversion costs

Convert once when possible. The cheapest assembly format need not be the best solve format.

$$
\boxed{
\text{assemble once}
\rightarrow
\text{convert once}
\rightarrow
\text{solve many}
}
$$

## 29. When direct sparse solves are useful

Use them for moderate problem sizes, multiple right-hand sides, or as a reference solution. Check fill-in and memory before scaling up.

## 30. When matrix-free methods help

They help when operator application is cheap and storage is costly. Preconditioning and boundary implementation remain central difficulties.

$$
Jv
$$

$$
\boxed{
\text{matrix entries}
\rightarrow
\text{matrix-vector action}
}
$$

## 31. Read a sparse calculation

Trace unknown ordering, assembly, format conversion, solver selection, preconditioner action, and residual checks.

## 32. Sparse-method map

Storage format and solver method solve different problems. Do not confuse a memory representation with a convergence guarantee.

| Purpose | SciPy |  
| --- | --- |  
| sparse diagonal matrix | `sparse.diags` |  
| COO assembly | `coo_matrix` |  
| CSR | `tocsr()` |  
| CSC | `tocsc()` |  
| identity | `sparse.eye` |  
| Kronecker product | `sparse.kron` |  
| sparse direct solve | `spsolve` |  
| reusable factorization | `factorized`, `splu` |  
| CG | `spla.cg` |  
| GMRES | `spla.gmres` |  
| BiCGSTAB | `spla.bicgstab` |  
| incomplete LU | `spilu` |  
| matrix-free operator | `LinearOperator` |  
| sparse symmetric eigen | `eigsh` |  
| sparse general eigen | `eigs` |

## 33. Continue to P9

Nonlinear equations replace a fixed linear operator with a residual that must be repeatedly evaluated and locally linearized.

$$
\boxed{
Ax=b
\rightarrow
F(x)=0
}
$$

## Supplement: independent numerical checks

These checks target mathematical invariants or an independent reference. They do not validate a wood constitutive law against observations. Tolerances belong to the stated example and tested environment.

In [25]:
# Direct, iterative, and matrix-free paths should solve the same SPD problem.
ntest=31
htest=1/(ntest+1)
Atest=sparse.diags([-np.ones(ntest-1),2*np.ones(ntest),-np.ones(ntest-1)],
                   [-1,0,1],format="csr")/htest**2
btest=np.sin(np.arange(1,ntest+1)*np.pi*htest)
xdirect=spla.spsolve(Atest,btest)
def positive_stencil(v):
    padded=np.pad(v,1)
    return (2*padded[1:-1]-padded[:-2]-padded[2:])/htest**2
optest=spla.LinearOperator(Atest.shape,matvec=positive_stencil,dtype=float)
xiter,status=spla.cg(optest,btest,rtol=1e-11,atol=0.0,maxiter=200)
assert status==0
assert np.linalg.norm(Atest@xiter-btest)/np.linalg.norm(btest)<1e-10
np.testing.assert_allclose(xiter,xdirect,rtol=1e-10,atol=1e-13)
print("Sparse direct, CG, and matrix-free comparison passed.")


Sparse direct, CG, and matrix-free comparison passed.


## Further work

1. Change one parameter or discretization choice and predict the effect before running.
2. Write the input and output shapes, units, and field locations.
3. Construct a deliberately invalid input and make the calculation report it clearly.
4. Separate an execution check from an accuracy check and from any physical claim.

## Primary documentation

- [SciPy sparse linear algebra](https://docs.scipy.org/doc/scipy/reference/sparse.linalg.html)

These are living documentation links. The accompanying requirements and verification report identify the versions actually executed for this edition.

## AI assistance and responsibility

The English adaptation, editorial supplementation, and code review were prepared with AI assistance. Wook Kang retains responsibility for reviewing the explanations, examples, and any subsequent research application. No new experimental validation is claimed.